# Padel Highlights - GPU Inference Pipeline

Este notebook está diseñado para ejecutar la inferencia de jugadores (YOLO) y bola (TrackNet) utilizando la GPU de Google Colab. Esto reducirá el tiempo de procesamiento de horas a minutos.

### 1. Montar Google Drive
Recomendamos subir los vídeos a una carpeta en Google Drive para que los resultados se guarden permanentemente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Configurar Entorno (Clonar e Instalar)
Este bloque se encarga de todo: clona el repo, actualiza el código, instala dependencias y configura el PYTHONPATH.

In [ ]:
import os
import sys

# 1. Clonar el repositorio si no existe
if not os.path.exists('padel-highlights'):
    !git clone https://github.com/rubenperezsoto/padel-highlights.git

# 2. Entrar en la carpeta
%cd /content/padel-highlights

# 3. Asegurar que tenemos la última versión
!git pull origin main

# 4. Instalar dependencias
!pip install ultralytics pandas torch torchvision opencv-python joblib scikit-learn torchsummary tensorboardX python-dotenv

# 5. Configurar PYTHONPATH para que Python encuentre el módulo 'src'
os.environ['PYTHONPATH'] = '/content/padel-highlights'
sys.path.append('/content/padel-highlights')

print("✅ Entorno configurado correctamente.")

### 3. Configurar TrackNetV2 y Pesos
Descargamos el repositorio de TrackNet y configuramos los pesos necesarios.

In [ ]:
# Crear directorios
!mkdir -p external/weights

# Clonar TrackNetV2-pytorch si no existe
if not os.path.exists('external/TrackNetV2-pytorch'):
    !git clone https://github.com/ChgygLin/TrackNetV2-pytorch external/TrackNetV2-pytorch

# Aplicar el parche para los pesos convertidos
%cd external/TrackNetV2-pytorch
!git apply tf2torch/diff.txt
%cd ../..

# Copiar pesos
!cp external/TrackNetV2-pytorch/tf2torch/track.pt external/weights/trackvnet.pt

### 4. Configurar Variables de Entorno para Modelos
Configuramos las rutas para que el código encuentre los modelos de TrackNet.

In [ ]:
import os
os.environ['TRACKNETV2_ROOT'] = '/content/padel-highlights/external/TrackNetV2-pytorch'
os.environ['TRACKNETV2_WEIGHTS'] = '/content/padel-highlights/external/weights/trackvnet.pt'

### 5. Ejecutar Inferencia
Cambia la ruta de `--video` por la ubicación de tu vídeo en Google Drive.

In [ ]:
# Ejemplo de ejecución
VIDEO_PATH = "/content/drive/MyDrive/padel/tu_video.mp4" # <--- CAMBIA ESTO
OUTPUT_PATH = "/content/drive/MyDrive/padel/ticks_resultado.parquet"

# Aseguramos que estamos en el directorio correcto
%cd /content/padel-highlights

!python3 -m src.padel.pipelines.run_inference_to_parquet \
    --video "{VIDEO_PATH}" \
    --output "{OUTPUT_PATH}"